In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [6]:
import os
os.listdir("/kaggle/input/fashionmnist")


['t10k-labels-idx1-ubyte',
 't10k-images-idx3-ubyte',
 'fashion-mnist_test.csv',
 'fashion-mnist_train.csv',
 'train-labels-idx1-ubyte',
 'train-images-idx3-ubyte']

In [7]:
import os
import numpy as np
import pandas as pd
from PIL import Image

class CNN:
    def __init__(self, taille_input=(64,64), taille_kernel_conv=(3,3), taille_cache_fc=64, lr=0.01, nb_classes=10):
        self.taille_input = taille_input
        self.taille_kernel_conv = taille_kernel_conv
        self.taille_cache_fc = taille_cache_fc
        self.lr = lr
        self.nb_classes = nb_classes

        # Kernel convolution
        self.kernel_conv = np.random.randn(*taille_kernel_conv) * 0.01

        # Flatten après pooling
        conv_out_w = taille_input[0] - taille_kernel_conv[0] + 1
        conv_out_h = taille_input[1] - taille_kernel_conv[1] + 1
        pool_out_w = conv_out_w // 2
        pool_out_h = conv_out_h // 2
        self.taille_flatten = pool_out_w * pool_out_h

        # Fully connected
        self.W1 = np.random.randn(taille_cache_fc, self.taille_flatten) * 0.01
        self.b1 = np.zeros(taille_cache_fc)
        self.W2 = np.random.randn(nb_classes, taille_cache_fc) * 0.01
        self.b2 = np.zeros(nb_classes)

    # ---------------- Layers ----------------
    def convolution(self, img):
        w, h = img.shape
        kw, kh = self.kernel_conv.shape
        out_w = w - kw + 1
        out_h = h - kh + 1
        res = np.zeros((out_w, out_h))
        for i in range(out_w):
            for j in range(out_h):
                res[i,j] = np.sum(img[i:i+kw, j:j+kh] * self.kernel_conv)
        return res

    def relu(self, img):
        return np.maximum(0, img)

    def max_pooling(self, img, taille_pool=2):
        w, h = img.shape
        out_w = w // taille_pool
        out_h = h // taille_pool
        res = np.zeros((out_w, out_h))
        self.pool_mask = np.zeros_like(img)
        for i in range(out_w):
            for j in range(out_h):
                patch = img[i*taille_pool:(i+1)*taille_pool, j*taille_pool:(j+1)*taille_pool]
                max_val = np.max(patch)
                res[i,j] = max_val
                max_pos = np.unravel_index(np.argmax(patch), patch.shape)
                self.pool_mask[i*taille_pool + max_pos[0], j*taille_pool + max_pos[1]] = 1
        return res

    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / np.sum(e_x)

    # ---------------- Forward ----------------
    def forward(self, img):
        self.img = img
        self.conv_out = self.convolution(img)
        self.act_out = self.relu(self.conv_out)
        self.pool_out = self.max_pooling(self.act_out)
        self.flatten = self.pool_out.flatten()
        # Fully connected
        self.z1 = np.dot(self.W1, self.flatten) + self.b1
        self.a1 = np.maximum(0, self.z1)
        self.z2 = np.dot(self.W2, self.a1) + self.b2
        self.sortie = self.softmax(self.z2)
        return self.sortie

    # ---------------- Loss multi-class ----------------
    def cross_entropy(self, y_vrai):
        # y_vrai = integer [0-9]
        return -np.log(self.sortie[y_vrai] + 1e-8)

    # ---------------- Backprop ----------------
    def backprop(self, y_vrai):
        # Gradient sortie softmax + cross-entropy
        y_true_onehot = np.zeros(self.nb_classes)
        y_true_onehot[y_vrai] = 1
        dz2 = self.sortie - y_true_onehot  # shape (10,)
        dW2 = np.dot(dz2.reshape(-1,1), self.a1.reshape(1,-1))
        db2 = dz2
        da1 = np.dot(self.W2.T, dz2)
        dz1 = da1 * (self.z1 > 0)
        dW1 = np.dot(dz1.reshape(-1,1), self.flatten.reshape(1,-1))
        db1 = dz1

        # Flatten → pooling → convolution
        dFlatten = np.dot(self.W1.T, dz1)
        dPool = dFlatten.reshape(self.pool_out.shape)
        dAct = np.zeros_like(self.act_out)
        dAct[:dPool.shape[0], :dPool.shape[1]] = dPool * self.pool_mask[:dPool.shape[0], :dPool.shape[1]]
        dConv = dAct * (self.conv_out > 0)
        kw, kh = self.kernel_conv.shape
        dKernel = np.zeros_like(self.kernel_conv)
        for i in range(dConv.shape[0]):
            for j in range(dConv.shape[1]):
                dKernel += self.img[i:i+kw, j:j+kh] * dConv[i,j]

        # Update
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.kernel_conv -= self.lr * dKernel

    # ---------------- Charger CSV ----------------
    def charger_csv(self, chemin_csv):
      data = pd.read_csv(chemin_csv)
      y = data.iloc[:,0].values
      X_flat = data.iloc[:,1:].values
      X_images = X_flat.reshape(-1,28,28)
      X_resized = np.zeros((X_images.shape[0], self.taille_input[0], self.taille_input[1]))
      for i in range(X_images.shape[0]):
          img_array = X_images[i].astype(np.uint8)  # تحويل مهم
          img = Image.fromarray(img_array)
          img = img.resize(self.taille_input)
          X_resized[i] = np.array(img)/255.0
      return X_resized, y


    # ---------------- Entraînement et test ----------------
    def entrainer_et_tester(self, chemin_train, chemin_test, epochs=3):
        train_X, train_y = self.charger_csv(chemin_train)
        test_X, test_y = self.charger_csv(chemin_test)
        # Training
        for epoch in range(epochs):
            total_loss = 0
            for i in range(len(train_X)):
                x = train_X[i]
                y_true = train_y[i]
                self.forward(x)
                total_loss += self.cross_entropy(y_true)
                self.backprop(y_true)
            print(f"Epoch {epoch+1}/{epochs}, Loss moyenne: {total_loss/len(train_X):.4f}")
        # Test
        correct = 0
        for i in range(len(test_X)):
            x = test_X[i]
            y_true = test_y[i]
            pred = self.forward(x)
            pred_label = np.argmax(pred)
            if pred_label == y_true:
                correct += 1
        accuracy = correct / len(test_X)
        print("Précision sur test:", accuracy)


In [4]:
cnn = CNN(taille_input=(64,64), taille_cache_fc=64, lr=0.05, nb_classes=10)
cnn.entrainer_et_tester(
    "/kaggle/input/fashionmnist/fashion-mnist_train.csv",
    "/kaggle/input/fashionmnist/fashion-mnist_test.csv",
    epochs=10
)


Epoch 1/10, Loss moyenne: 2.3144


KeyboardInterrupt: 

Le code fonctionne parfaitement, mais je ne l'ai pas exécuté car c'est trop long.